### RAGAS

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import pandas as pd

df = pd.read_csv("data/ragas_synthetic_dataset.csv")
df.head(2)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the key components of the executive o...,"['1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하...",The executive order signed by President Biden ...,AI Safety Researcher,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
1,G7가 히로시마 AI 프로세스를 통해 AI 기업을 위한 국제 행동강령을 만든 이유는...,"['SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스...",G7은 히로시마 AI 프로세스를 통해 AI 기업을 대상으로 AI 위험 식별과 완화를...,AI Governance Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [4]:
from datasets import Dataset

test_dataset = Dataset.from_pandas(df)
test_dataset

Dataset({
    features: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name'],
    num_rows: 10
})

In [5]:
import ast

def convert_to_list(example):
    value = example["reference_contexts"]
    if isinstance(value, str):
        value = ast.literal_eval(value)

    if not isinstance(value, list):
        raise TypeError(f"reference_contexts must be a list, got {type(value).__name__}")

    return {"reference_contexts": value}

test_dataset = test_dataset.map(convert_to_list)

print(test_dataset["reference_contexts"])
print("---")
print(test_dataset[0]["reference_contexts"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Column([['1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표\nKEY Contents\nn 미국 바이든 대통령이 ‘안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령’에 서명하고\n광범위한 행정 조치를 명시\nn 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자\n보호 △노동자 지원 △혁신과 경쟁 촉진 △국제협력을 골자로 함\n£바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 추진\nn 미국 바이든 대통령이 2023년 10월 30일 연방정부 차원에서 안전하고 신뢰할 수 있는 AI 개발과\n사용을 보장하기 위한 행정명령을 발표\n∙ 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자 보호\n△노동자 지원 △혁신과 경쟁 촉진 △국제협력에 관한 내용을 포괄\nn (AI 안전과 보안 기준) 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한\n주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및\nAI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진\n∙ △1026 플롭스(FLOPS, Floating Point Operation Per Second)를 초과하는 컴퓨팅 성능 또는 생물학적\n서열 데이터를 주로 사용하고 1023플롭스를 초과하는 컴퓨팅 성능을 사용하는 모델 △단일 데이터센터에서\n1,000Gbit/s 이상의 네트워킹으로 연결되며 AI 훈련에서 이론상 최대 1020 플롭스를 처리할 수 있는\n컴퓨팅 용량을 갖춘 컴퓨팅 클러스터가 정보공유 요구대상\nn (형평성과 시민권 향상) 법률, 주택, 보건 분야에서 AI의 무책임한 사용으로 인한 차별과 편견 및 기타\n문제를 방지하는 조치를 확대\n∙ 형사사법 시스템에서 AI 사용 모범사례를 개발하고, 주택 임대 시 AI 알고리즘 차별을 막

In [ ]:
# !uv add pymupdf

In [ ]:
# !uv add faiss-cpu

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

loader = PyMuPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

retriever = vectorstore.as_retriever()

prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

In [11]:
batch_dataset = list(test_dataset["user_input"])
batch_dataset[:3]

['What are the key components of the executive order signed by President Biden on October 30, 2023, regarding the development and use of safe and trustworthy AI?',
 'G7가 히로시마 AI 프로세스를 통해 AI 기업을 위한 국제 행동강령을 만든 이유는 뭐고, 그 내용은 어떤거야?',
 '프랑스는 AI 안전성 정상회의에서 어떤 역할을 할 것인가요?']

In [12]:
retrieved_documents = retriever.batch(batch_dataset)
retrieved_contexts = [
    [doc.page_content for doc in documents]
    for documents in retrieved_documents
]

responses = chain.batch(batch_dataset)
responses

['The key components of the executive order signed by President Biden on October 30, 2023, regarding the development and use of safe and trustworthy AI include:\n\n1. Establishing safety and security standards for AI.\n2. Ensuring privacy protection.\n3. Enhancing equity and civil rights.\n4. Protecting consumers.\n5. Supporting workers.\n6. Promoting innovation and competition.\n7. Encouraging international cooperation.\n\nThe order requires companies developing powerful AI systems to share safety test results and key information with the U.S. government, and it aims to establish standards for the safety and reliability of AI systems, as well as for labeling AI-generated content.',
 'G7는 히로시마 AI 프로세스를 통해 AI 기업을 위한 국제 행동강령을 만든 이유는 첨단 AI 시스템의 개발과 관련된 위험을 식별하고 완화하기 위해 자발적인 채택을 권고하기 위함입니다. 이 행동강령은 AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성의 보장, 정보 공유와 이해관계자 간 협력, 보안 통제, 콘텐츠 인증과 출처 확인 등의 조치를 요구합니다.\n\n주요 내용은 다음과 같습니다:\n- AI 시스템의 개발 과정에서 위험을 평가하고 완화하는 조치를 채택\n- AI 시스템의 성능과 한계를 공개하고 적절한 사용 영역을 알리는 방

In [15]:
for column_name, values in {
    "retrieved_contexts": retrieved_contexts,
    "response": responses,
}.items():
    if column_name in test_dataset.column_names:
        test_dataset = test_dataset.remove_columns(column_name)
    test_dataset = test_dataset.add_column(column_name, values)

print(test_dataset.column_names)
test_dataset.select_columns([
    "user_input", "retrieved_contexts", "responses", "reference"
])

['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name', 'responses', 'retrieved_contexts', 'response']


Dataset({
    features: ['user_input', 'retrieved_contexts', 'responses', 'reference'],
    num_rows: 10
})

In [16]:
from ragas import RunConfig, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import(
    AnswerRelevancy,
    Faithfulness,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        timeout=60,
        max_retries=1,
    )
)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(
        model="text-embedding-3-small",
        timeout=60,
        max_retries=1,
    )
)

metrics = [
    Faithfulness(),
    AnswerRelevancy(strictness=1),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
]

result = evaluate(
    dataset=test_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(
        timeout=60,
        max_retries=2,
        max_wait=10,
        max_workers=4,
    ),
    batch_size=2,
    raise_exceptions=True,
)

result

C:\Users\user\AppData\Local\Temp\ipykernel_4544\1899244835.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import(
C:\Users\user\AppData\Local\Temp\ipykernel_4544\1899244835.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import(
C:\Users\user\AppData\Local\Temp\ipykernel_4544\1899244835.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import(
C:\Users\user\AppData\Local

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Batch 1/20:   0%|          | 0/2 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001B3C4E127C0> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001B3C4E127C0> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cann

{'faithfulness': 0.8333, 'answer_relevancy': 0.7231, 'llm_context_precision_with_reference': 0.9250, 'context_recall': 0.9750}

In [17]:
result_df = result.to_pandas()
result_df.head()

,user_input,retrieved_contexts,reference_contexts,response,reference,persona_name,query_style,query_length,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What are the key components of the executive o...,[1. 정책/법제 \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...",The key components of the executive order sign...,The executive order signed by President Biden ...,AI Safety Researcher,WEB_SEARCH_LIKE,LONG,1.000000,0.924848,1.00,0.75
1,G7가 히로시마 AI 프로세스를 통해 AI 기업을 위한 국제 행동강령을 만든 이유는...,"[SPRi AI Brief | \n2023-12월호\n2\nG7, 히로시마 AI ...","[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...",G7는 히로시마 AI 프로세스를 통해 AI 기업을 위한 국제 행동강령을 만든 이유는...,G7은 히로시마 AI 프로세스를 통해 AI 기업을 대상으로 AI 위험 식별과 완화를...,AI Governance Analyst,POOR_GRAMMAR,LONG,1.000000,0.635504,1.00,1.00
2,프랑스는 AI 안전성 정상회의에서 어떤 역할을 할 것인가요?,[1. 정책/법제 \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,프랑스는 AI 안전성 정상회의에서 1년 후 대면 정상회의를 개최할 예정입니다.,프랑스 정부는 1년 후 대면 정상회의를 개최할 예정입니다.,AI Safety Researcher,POOR_GRAMMAR,LONG,0.000000,0.544051,0.50,1.00
3,사라 앤더슨이 저작권 소송에서 어떤 역할을 했는지 설명해 주세요.,"[SPRi AI Brief | \n2023-12월호\n4\n미국 법원, 예술가들이...","[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...","사라 앤더슨은 저작권 소송에서 원고 중 한 명으로, 미드저니, 스태빌리티AI, 디비...","사라 앤더슨은 2023년 1월에 미드저니, 스태빌리티AI, 디비언트아트를 상대로 저...",AI Governance Analyst,MISSPELLED,LONG,0.833333,0.382584,1.00,1.00
4,FTC는 2023년 5월 31일에 어떤 조치를 취했나요?,[데이터에 대한 독점 라이선스를 확보해 시장 지배력을 더욱 강화할 수 있다는 우려도...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,FTC는 2023년 5월 31일에 동의를 받지 않고 어린이들의 음성과 위치 정보를 ...,FTC는 2023년 5월 31일 동의를 받지 않고 어린이들의 음성과 위치 정보를 활...,AI Safety Researcher,POOR_GRAMMAR,MEDIUM,1.000000,0.644498,0.75,1.00
